# Module 12 — RAG v2: Production Retrieval

Companion notebook to [`docs/modules/12_rag_v2_production_retrieval.md`](../docs/modules/12_rag_v2_production_retrieval.md).

Evolves Module 11's naive RAG into production-grade retrieval: semantic/structural/parent-child
chunking, query rewriting/multi-query/HyDE, ACL-aware and time-aware retrieval, a real heuristic
reranker, budget-based context packing with lost-in-the-middle mitigation, source-level
citations, and incremental indexing. Every stage runs for real except the final answer
generation call (`FakeRuntime`, wired against Module 6's `LLMRuntime` protocol unchanged for a
real adapter later) and the cross-encoder reranker (honest-skip, needs downloaded model
weights).


In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "packages"))
sys.path.insert(0, str(REPO_ROOT / "scripts" / "module_12"))
CORPUS_DIR = REPO_ROOT / "datasets" / "rag_docs" / "nimbus_handbook"
print(f"Repo root: {REPO_ROOT}")


Repo root: /Users/bhakti/workspace/local_ai_app


## 1. Semantic chunking vs. fixed-size chunking — real, different boundaries

In [2]:
from local_ai_rag.chunkers.document_chunker import chunk_document
from local_ai_rag.chunkers.semantic_chunker import chunk_semantically
from local_ai_rag.embeddings.fake import FakeEmbedder
from local_ai_rag.loaders.markdown_loader import load_markdown_directory

embedder = FakeEmbedder(dimensions=64)
documents = load_markdown_directory(CORPUS_DIR)
sample_doc = next(d for d in documents if d.doc_id == "password_reset")

fixed_chunks = chunk_document(sample_doc, max_chars=150)
semantic_chunks = await chunk_semantically(sample_doc, embedder, similarity_threshold=0.15)

print(f"Fixed-size (150 chars): {len(fixed_chunks)} chunks")
print(f"Semantic:               {len(semantic_chunks)} chunks")
for c in semantic_chunks:
    print(f"  [{c.chunk_index}] {c.text[:80]}...")


Fixed-size (150 chars): 6 chunks
Semantic:               4 chunks
  [0] If you forget your Nimbus password, go to the sign-in page and select "Forgot pa...
  [1] After resetting your password, all active sessions on all devices are signed out...
  [2] Nimbus does not send password reset links via SMS....
  [3] If you no longer have access to the email address on file, contact support with ...


## 2. Table/code-aware chunking — structure is never split

In [3]:
from local_ai_rag.chunkers.structural_chunker import chunk_preserving_structure
from local_ai_rag.loaders.markdown_loader import Document

table_doc = Document(
    doc_id="pricing_table",
    source_path="demo",
    title="Pricing",
    text=(
        "Here is the pricing table.\n\n"
        "| Plan | Price |\n|------|-------|\n| Free | $0 |\n| Personal | $6.99 |\n\n"
        "Trailing paragraph after the table."
    ),
)
structural_chunks = chunk_preserving_structure(table_doc, max_chars=20)
for c in structural_chunks:
    print(f"structural={c.contains_structural_block}  {c.text!r}")


structural=False  'Here is the pricing'
structural=False  'table.'
structural=True  '| Plan | Price |\n|------|-------|\n| Free | $0 |\n| Personal | $6.99 |'
structural=False  'Trailing paragraph'
structural=False  'after the table.'


## 3. Parent-child retrieval — Lab 1 (index small, retrieve big)

In [4]:
from parent_child_demo import build_parent_child_retriever, QUESTION as PARENT_CHILD_QUESTION

retriever, pc_documents, pc_index = await build_parent_child_retriever()
results = await retriever.retrieve(PARENT_CHILD_QUESTION, k=3)
print(f"{len(pc_index.parents)} parent chunks, {len(pc_index.children)} child chunks indexed")
for r in results:
    print(f"{r.best_child_score:.3f}  {r.parent_id}  ({len(r.text)} chars)")


20 parent chunks, 100 child chunks indexed
0.420  account_creation::0  (693 chars)
0.406  file_sharing::0  (493 chars)
0.362  security_incident_response::0  (597 chars)


## 4. Query rewriting, multi-query retrieval, and HyDE — mechanically real, generation honest-skip

In [5]:
from local_ai_core.runtimes.fake import FakeRuntime
from local_ai_rag.retrievers.naive_retriever import NaiveRetriever
from local_ai_rag.retrievers.query_expansion import hyde_retrieve, rewrite_query
from local_ai_rag.stores.numpy_store import NumpyVectorStore
from local_ai_rag.chunkers.document_chunker import chunk_documents

store = NumpyVectorStore()
chunks = chunk_documents(documents, max_chars=500)
vectors = await embedder.embed_documents([c.text for c in chunks])
for chunk, vector in zip(chunks, vectors):
    await store.add(chunk.chunk_id, chunk.text, vector, metadata={"doc_id": chunk.doc_id})

retriever = NaiveRetriever(embedder, store)
runtime = FakeRuntime(default_response="How do I reset my Nimbus account password?")

rewritten = await rewrite_query("forgot my password help", runtime, model="fake-model")
print("Rewritten query:", rewritten)

hyde_results = await hyde_retrieve(
    "how long until my reset link expires", embedder, store,
    FakeRuntime(default_response="Password reset links expire in fifteen minutes for security."),
    model="fake-model", k=2,
)
print("HyDE top result:", hyde_results[0].doc_id if hyde_results else None)


Rewritten query: How do I reset my Nimbus account password?
HyDE top result: api_rate_limits::0


## 5. Time-aware retrieval — recency boost

In [6]:
from datetime import datetime, timedelta, timezone

from local_ai_rag.embeddings.embedder import SearchResult
from local_ai_rag.retrievers.time_aware import apply_recency_boost

now = datetime.now(timezone.utc)
candidates = [
    SearchResult(doc_id="old_but_relevant", score=1.0, text="t", metadata={"created_at": (now - timedelta(days=1)).isoformat()}),
    SearchResult(doc_id="recent_but_weak", score=0.05, text="t", metadata={"created_at": now.isoformat()}),
    SearchResult(doc_id="ancient", score=1.0, text="t", metadata={"created_at": (now - timedelta(days=365)).isoformat()}),
]
boosted = apply_recency_boost(candidates, half_life_days=30.0, now=now)
for r in boosted:
    print(f"{r.score:.4f}  {r.doc_id}")


0.9772  old_but_relevant
0.0500  recent_but_weak
0.0002  ancient


## 6. ACL filtering, reranking, context packing, source-level citations — Labs 2-5

`security_incident_response` is tagged as a restricted internal-only document mixed into the
handbook. A citation-grounding check catches two real, different problems: an ACL leak (citing
a document the user was never shown) and a packing-related drop (citing a document that exists
but didn't survive reranking/packing).

In [7]:
from reranking_demo import run_lab as run_reranking_lab, result_to_markdown as reranking_to_markdown

reranking_result = await run_reranking_lab()
print(reranking_to_markdown(reranking_result))


# Labs 2-5 - ACL filtering, reranking, context packing, source-level citations

- Question: How do I reset my password and what should I do if I suspect unauthorized access?
- Low-clearance user sees documents from: ['api_rate_limits', 'file_upload_limits', 'invoices_and_receipts', 'password_reset', 'supported_regions']
- High-clearance user sees documents from: ['api_rate_limits', 'file_upload_limits', 'invoices_and_receipts', 'security_incident_response', 'supported_regions']
- Low-clearance source citations: ['password_reset', 'security_incident_response'] (grounded: False)
- High-clearance source citations: ['password_reset', 'security_incident_response'] (grounded: False)
- Chunks packed with a generous budget: 5
- Chunks packed with a tight budget: 1



## 7. Lost-in-the-middle mitigation — highest-relevance chunks at both edges

In [8]:
from local_ai_rag.context_packers.budget_packer import order_for_generation

ranked = [SearchResult(doc_id=f"rank-{i}", score=1.0 - i * 0.1, text="t", metadata={}) for i in range(5)]
ordered = order_for_generation(ranked)
print("Input order (by relevance):", [r.doc_id for r in ranked])
print("Output order (edges first):", [r.doc_id for r in ordered])


Input order (by relevance): ['rank-0', 'rank-1', 'rank-2', 'rank-3', 'rank-4']
Output order (edges first): ['rank-0', 'rank-2', 'rank-4', 'rank-3', 'rank-1']


## 8. Incremental indexing — Lab 6

In [9]:
from incremental_indexing_demo import run_lab as run_incremental_lab, result_to_markdown as incremental_to_markdown

incremental_result = await run_incremental_lab()
print(incremental_to_markdown(incremental_result))


# Lab 6 - incremental indexing

- First sync: 20 documents -> 38 chunks, 20 embed_documents() call(s)
- Second sync (1 document edited, 1 removed, 18 unchanged):
  - added: []
  - updated: ['password_reset']
  - deleted: ['supported_regions']
  - embed_documents() calls triggered: 1 (not re-embedding the 18 unchanged documents)
- Chunks after second sync: 36



## 9. Cross-encoder reranker — honest skip

Built and fully unit-tested (`cross_encoder_reranker.py`, injected fake `load_fn`/`score_fn`,
same lazy-import/DI pattern as Module 9's `SentenceTransformersEmbedder`), but a real
cross-encoder model needs downloaded weights this machine doesn't have.

In [10]:
import importlib.util

sentence_transformers_installed = importlib.util.find_spec("sentence_transformers") is not None
print(f"sentence-transformers installed: {sentence_transformers_installed}")
print("Skipping a real cross-encoder rerank: no downloaded model on this machine (see the machine note).")


sentence-transformers installed: False
Skipping a real cross-encoder rerank: no downloaded model on this machine (see the machine note).
